# Portfolio Optimization: The Value of Stochastic Programming

This notebook illustrates the importance of **stochastic programming** compared to a classical deterministic approach (based on expected values) for stock portfolio optimization.

## 1. The Problem
We have an initial capital $W = 1$ to invest in $N$ stocks. Each stock $i$ has a random return $r_i$.
Our goal is to maximize the expected value of the portfolio, but with a **penalty** if the portfolio's value falls below a target return $T$ (downside risk management).

**Deterministic Model (Expected Value - EV):**
We replace the random returns with their historical mean $\mu_i = \mathbb{E}[r_i]$.
The model assumes that the future will perfectly match the historical average.

**Stochastic Model (Stochastic Program - SP):**
We use a set of scenarios (here, historical daily returns) to represent uncertainty. The model optimizes decisions by considering all possible scenarios.

We will use the `YFinance.jl` library to fetch real data and `JuMP.jl` for optimization.

In [ ]:
# Install necessary packages (uncomment if needed)
# import Pkg
# Pkg.add(["JuMP", "HiGHS", "DataFrames", "YFinance", "Statistics", "Plots"])

In [ ]:
using JuMP
using HiGHS
using YFinance
using DataFrames
using Statistics
using Plots

## 2. Data Fetching with Yahoo Finance
We will download the adjusted close prices of a few tech stocks to simulate our scenarios.

In [ ]:
tickers = ["AAPL", "MSFT", "GOOG", "AMZN"]
start_date = "2021-01-01"
end_date = "2023-12-31"

prices = DataFrame()

for ticker in tickers
    # Fetch historical prices
    data = get_prices(ticker, startdt=start_date, enddt=end_date)
    # Extract adjusted close prices and dates
    if isempty(prices)
        prices.Date = data["timestamp"]
    end
    prices[!, ticker] = data["adjclose"]
end

first(prices, 5)

We now compute the daily returns $r = \frac{P_t}{P_{t-1}}$. Each day will represent an equiprobable "scenario" in our stochastic model.

In [ ]:
# Calculate daily returns (gross returns)
returns = DataFrame()
returns.Date = prices.Date[2:end]

for ticker in tickers
    returns[!, ticker] = prices[2:end, ticker] ./ prices[1:end-1, ticker]
end

# Returns matrix where each row is a scenario s, and each column an asset i
R = Matrix(returns[:, tickers])
num_scenarios, num_assets = size(R)

# Calculate mean returns for the deterministic model
mu = mean(R, dims=1)[:] 

println("Number of scenarios (days): ", num_scenarios)
println("Average daily returns: ")
for (i, t) in enumerate(tickers)
    println("$t : $(round(mu[i], digits=4))")
end

## 3. Optimization Parameters
We aim to invest our budget while minimizing the risk of falling below a target return $T$.

In [ ]:
W = 1.0       # Initial budget
T = 1.002     # Target final wealth (e.g., +0.2% per day)
q = 10.0      # Penalty for each unit of wealth below T

## 4. Deterministic Model (Expected Value - EV)
The deterministic model solves the problem using only the expected return $\mu$.

$\max_{x} \sum_{i} \mu_i x_i - q \cdot \max(0, T - \sum_{i} \mu_i x_i)$
subject to: $\sum_{i} x_i = W, \quad x_i \ge 0$

In [ ]:
model_EV = Model(HiGHS.Optimizer)
set_silent(model_EV)

@variable(model_EV, x_ev[1:num_assets] >= 0)  # Invested fractions
@variable(model_EV, y_ev >= 0)                # Variable to model max(0, ...)

# Budget constraint
@constraint(model_EV, sum(x_ev) == W)

# Constraint for the shortfall relative to the target T
@constraint(model_EV, y_ev >= T - sum(mu[i] * x_ev[i] for i in 1:num_assets))

# Objective function: Maximize expected final wealth minus the penalty
@objective(model_EV, Max, sum(mu[i] * x_ev[i] for i in 1:num_assets) - q * y_ev)

optimize!(model_EV)

x_ev_sol = value.(x_ev)
println("Deterministic Allocation (EV):")
for (i, t) in enumerate(tickers)
    println("$t : $(round(x_ev_sol[i] * 100, digits=2)) %")
end


The deterministic model is blind to variance risk. It usually invests everything in the stock with the highest expected return that meets the target or provides the best average trade-off.

## 5. Stochastic Model (Stochastic Program - SP)
The stochastic model takes all possible scenarios into account.

$\max_{x, y_s} \frac{1}{S} \sum_{s=1}^S \left( \sum_{i} R_{s,i} x_i - q \cdot y_s \right)$
subject to:
$\sum_{i} x_i = W, \quad x_i \ge 0$
$y_s \ge T - \sum_{i} R_{s,i} x_i, \quad y_s \ge 0 \quad \forall s=1 \dots S$

In [ ]:
model_SP = Model(HiGHS.Optimizer)
set_silent(model_SP)

@variable(model_SP, x_sp[1:num_assets] >= 0)
@variable(model_SP, y_sp[1:num_scenarios] >= 0)

@constraint(model_SP, sum(x_sp) == W)

# Shortfall constraints for each scenario
for s in 1:num_scenarios
    @constraint(model_SP, y_sp[s] >= T - sum(R[s, i] * x_sp[i] for i in 1:num_assets))
end

# Objective function: Expected (mean across scenarios) final wealth minus penalty
@objective(model_SP, Max, (1/num_scenarios) * sum( sum(R[s,i] * x_sp[i] for i in 1:num_assets) - q * y_sp[s] for s in 1:num_scenarios ))

optimize!(model_SP)

x_sp_sol = value.(x_sp)
println("Stochastic Allocation (SP):")
for (i, t) in enumerate(tickers)
    println("$t : $(round(x_sp_sol[i] * 100, digits=2)) %")
end

RP = objective_value(model_SP)
println("\nSP Objective Value (Recourse Problem - RP): ", round(RP, digits=5))

The stochastic model diversifies the investment. It is aware that putting all eggs in one basket generates too many scenarios with heavy penalties.

## 6. The Value of Stochastic Solution (VSS)
To measure the importance of stochastic programming, we calculate the **EEV (Expected result of using the EV solution)**. 
This consists of taking the decision from the deterministic model (`x_ev_sol`) and evaluating it on the actual scenarios (our stochastic objective function).

The VSS is the difference: $VSS = RP - EEV$. The larger the VSS, the more crucial it is to account for uncertainty.

In [ ]:
# Evaluate EV allocation on the actual scenarios
EEV = 0.0
for s in 1:num_scenarios
    wealth = sum(R[s, i] * x_ev_sol[i] for i in 1:num_assets)
    penalty = max(0, T - wealth)
    EEV += (wealth - q * penalty)
end
EEV = EEV / num_scenarios

VSS = RP - EEV

println("RP (Recourse Problem - expected gain with SP approach): ", round(RP, digits=5))
println("EEV (Expected gain if we apply the EV solution)         : ", round(EEV, digits=5))
println("----------------------------------------------------------------")
println("VSS (Value of Stochastic Solution)                      : ", round(VSS, digits=5))

if VSS > 1e-6
    println("\nConclusion: The stochastic approach brings clear added value! Deterministic optimization is too risky here.")
else
    println("\nConclusion: The VSS is negligible, uncertainty does not heavily impact the decision quality in this specific case.")
end

## 7. Simulated Performance Visualization (Simplified Backtest)
Let's compare the daily return distribution of both portfolios.

In [ ]:
port_ev_returns = [sum(R[s, i] * x_ev_sol[i] for i in 1:num_assets) for s in 1:num_scenarios]
port_sp_returns = [sum(R[s, i] * x_sp_sol[i] for i in 1:num_assets) for s in 1:num_scenarios]

histogram(port_ev_returns, alpha=0.5, label="EV Portfolio (Deterministic)", bins=50, title="Daily Returns Distribution", xlabel="Gross Return", ylabel="Frequency")
histogram!(port_sp_returns, alpha=0.5, label="SP Portfolio (Stochastic)", bins=50)
vline!([T], label="Target T ($T)", lw=2, color=:red, linestyle=:dash)